# Panipokhari Urban Heat, Vegetation & Greening-Priority Analysis

**Portfolio project | Remote Sensing • GIS • Raster Analysis • QA/QC**

This notebook presents a cleaned, reproducible version of a Panipokhari, Kathmandu remote-sensing workflow. It combines Sentinel-2 vegetation/built-up indicators with Landsat 9 land-surface temperature (LST) to identify areas with higher combined heat exposure and greening need.

### What this demonstrates
- Reproducible raster workflows with `rasterio`, `geopandas`, `numpy`, and `pandas`
- NDVI and NDBI calculation
- LST conversion and Landsat QA masking
- Multi-class raster classification
- Raster alignment/resampling across 10 m and 30 m products
- A weighted spatial-priority model
- Pixel-level QA and **100% reconstruction validation**
- Transparent documentation of thresholds, formulas and assumptions.



## 1. Project structure

The notebook expects a repository structure similar to:

```text
panipokhari-remote-sensing/
├── notebooks/
│   └── panipokhari_remote_sensing.ipynb
├── data/
│   ├── panipokhari_analysis.gpkg
│   ├── sentinel2/
│   │   ├── B04_red.tif
│   │   ├── B08_nir.tif
│   │   ├── B11_swir.tif
│   │   └── SCL.tif
│   └── landsat9/
│       ├── LC09_L2SP_141041_20260420_20260421_02_T1_ST_B10.TIF
│       └── LC09_L2SP_141041_20260420_20260421_02_T1_QA_PIXEL.TIF
└── outputs/
    └── ...
```

The code below uses only `PROJECT_DIR`, `DATA_DIR`, and `OUTPUT_DIR` for file locations.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling

# -------------------------------------------------------------------
# Reproducible project paths — no machine-specific absolute paths.
# -------------------------------------------------------------------
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 2. Study area and source imagery

**Study area:** Panipokhari, Kathmandu, Nepal.

The  workflow uses:
- **Sentinel-2 L2A:** `S2A_45RUL_20260413_0_L2A` for 10 m vegetation/built-up analysis
- **Landsat 9 Collection 2 Level-2:** `LC09_L2SP_141041_20260420_20260421_02_T1` for LST

The Sentinel-2 AOI is projected to **EPSG:32645 (UTM Zone 45N)** for metric raster analysis.

Note: The notebook focuses on the analytical workflow rather than repeating exploratory scene-search/download/debugging steps.


In [ ]:
# Study area
AOI_PATH = DATA_DIR / "panipokhari_analysis.gpkg"

aoi = gpd.read_file(AOI_PATH, layer="study_area")
aoi_utm = aoi.to_crs("EPSG:32645")

print("Features:", len(aoi))
print("Input CRS:", aoi.crs)
print("Analysis CRS:", aoi_utm.crs)
print("Bounds:", aoi_utm.total_bounds)

In [ ]:
# Input raster locations
S2_RED = DATA_DIR / "sentinel2" / "B04_red.tif"
S2_NIR = DATA_DIR / "sentinel2" / "B08_nir.tif"
S2_SWIR = DATA_DIR / "sentinel2" / "B11_swir.tif"
S2_SCL = DATA_DIR / "sentinel2" / "SCL.tif"

L9_ST_B10 = DATA_DIR / "landsat9" / "LC09_L2SP_141041_20260420_20260421_02_T1_ST_B10.TIF"
L9_QA_PIXEL = DATA_DIR / "landsat9" / "LC09_L2SP_141041_20260420_20260421_02_T1_QA_PIXEL.TIF"

for p in [AOI_PATH, S2_RED, S2_NIR, S2_SWIR, S2_SCL, L9_ST_B10, L9_QA_PIXEL]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

## 3. Clip Sentinel-2 inputs to the study area

B04, B08, B11 and SCL are clipped to the study-area geometry. B04/B08 provide the 10 m red/NIR inputs; B11 is the SWIR input used for NDBI and is later aligned to the 10 m grid.



In [ ]:
aoi_shapes = [geom.__geo_interface__ for geom in aoi_utm.geometry]

def clip_raster(src_path, out_path, shapes, resampling=None):
    with rasterio.open(src_path) as src:
            clipped, transform = mask(src, shapes, crop=True)
        profile = src.profile.copy()
        profile.update(
            height=clipped.shape[1],
            width=clipped.shape[2],
            transform=transform
        )
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(clipped)

clip_paths = {
    "red": OUTPUT_DIR / "panipokhari_red.tif",
    "nir": OUTPUT_DIR / "panipokhari_nir.tif",
    "swir": OUTPUT_DIR / "panipokhari_B11_clipped.tif",
    "scl": OUTPUT_DIR / "panipokhari_scl.tif",
}

for name, src in {
    "red": S2_RED,
    "nir": S2_NIR,
    "swir": S2_SWIR,
    "scl": S2_SCL,
}.items():
    clip_raster(src, clip_paths[name], aoi_shapes)

print("Sentinel-2 clipping complete.")

## 4. NDVI — vegetation condition

NDVI is calculated as:

\[
NDVI = \frac{NIR - Red}{NIR + Red}
\]

For Sentinel-2, the workflow uses **B08 (NIR)** and **B04 (Red)**.

### Vegetation classes
| Class | NDVI rule | Interpretation used in this project |
|---|---|---|
| 1 | NDVI < 0.20 | Lower vegetation |
| 2 | 0.20 ≤ NDVI ≤ 0.40 | Moderate vegetation |
| 3 | NDVI > 0.40 | Higher vegetation |

These thresholds are validated later by pixel reconstruction.


In [ ]:
red_path = clip_paths["red"]
nir_path = clip_paths["nir"]

with rasterio.open(red_path) as src:
    red = src.read(1).astype("float32")
    ndvi_profile = src.profile.copy()

with rasterio.open(nir_path) as src:
    nir = src.read(1).astype("float32")

denominator = nir + red
ndvi = np.where(denominator == 0, np.nan, (nir - red) / denominator)

ndvi_path = OUTPUT_DIR / "panipokhari_ndvi.tif"
ndvi_out = np.where(np.isnan(ndvi), -9999, ndvi).astype("float32")

ndvi_profile.update(dtype="float32", count=1, nodata=-9999)
with rasterio.open(ndvi_path, "w", **ndvi_profile) as dst:
    dst.write(ndvi_out, 1)

print("NDVI range:", np.nanmin(ndvi), "to", np.nanmax(ndvi))
print("Mean NDVI:", np.nanmean(ndvi))

In [ ]:
# Align Sentinel-2 SCL to the 10 m NDVI grid.
scl_10m_path = OUTPUT_DIR / "panipokhari_scl_10m.tif"

with rasterio.open(ndvi_path) as ref, rasterio.open(clip_paths["scl"]) as src:
    scl_10m = np.zeros((ref.height, ref.width), dtype=np.uint8)
    reproject(
        source=rasterio.band(src, 1),
        destination=scl_10m,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref.transform,
        dst_crs=ref.crs,
        resampling=Resampling.nearest
    )
    profile = ref.profile.copy()
    profile.update(dtype="uint8", nodata=0)

with rasterio.open(scl_10m_path, "w", **profile) as dst:
    dst.write(scl_10m, 1)


ndvi_masked_path = OUTPUT_DIR / "panipokhari_ndvi_masked.tif"
ndvi_masked = np.where(scl_10m == 0, -9999, ndvi_out).astype("float32")

profile = rasterio.open(ndvi_path).profile.copy()
profile.update(dtype="float32", nodata=-9999)
with rasterio.open(ndvi_masked_path, "w", **profile) as dst:
    dst.write(ndvi_masked, 1)

valid = ndvi_masked != -9999
print("Valid NDVI pixels:", valid.sum())

In [ ]:
# Vegetation classification
veg_class_path = OUTPUT_DIR / "panipokhari_vegetation_class.tif"
veg_class = np.zeros(ndvi_masked.shape, dtype=np.uint8)
valid = ndvi_masked != -9999

veg_class[valid & (ndvi_masked < 0.20)] = 1
veg_class[valid & (ndvi_masked >= 0.20) & (ndvi_masked <= 0.40)] = 2
veg_class[valid & (ndvi_masked > 0.40)] = 3

profile = rasterio.open(ndvi_path).profile.copy()
profile.update(dtype="uint8", nodata=0)
with rasterio.open(veg_class_path, "w", **profile) as dst:
    dst.write(veg_class, 1)

print("Vegetation class counts:", dict(zip(*np.unique(veg_class, return_counts=True))))

## 5. NDBI — built-up intensity

NDBI is calculated as:

\[
NDBI = \frac{SWIR - NIR}{SWIR + NIR}
\]

The workflow resamples Sentinel-2 B11 to the 10 m B08 grid using **bilinear interpolation** before calculating NDBI.

### Built-up classes
| Class | NDBI rule |
|---|---|
| 1 | NDBI < 0.0572 |
| 2 | 0.0572 ≤ NDBI ≤ 0.1057 |
| 3 | NDBI > 0.1057 |



In [ ]:
# Align 20 m B11 to the 10 m NIR grid.
b11_10m_path = OUTPUT_DIR / "panipokhari_B11_10m.tif"

with rasterio.open(nir_path) as ref, rasterio.open(clip_paths["swir"]) as src:
    b11_10m = np.zeros((ref.height, ref.width), dtype="float32")
    reproject(
        source=rasterio.band(src, 1),
        destination=b11_10m,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref.transform,
        dst_crs=ref.crs,
        resampling=Resampling.bilinear
    )
    profile = ref.profile.copy()
    profile.update(dtype="float32", nodata=0)

with rasterio.open(b11_10m_path, "w", **profile) as dst:
    dst.write(b11_10m, 1)

# NDBI
with rasterio.open(nir_path) as src:
    nir = src.read(1).astype("float32")
    ndbi_profile = src.profile.copy()

denominator = b11_10m + nir
ndbi = np.where(denominator == 0, np.nan, (b11_10m - nir) / denominator)

ndbi_path = OUTPUT_DIR / "panipokhari_ndbi.tif"
ndbi_out = np.where(np.isnan(ndbi), -9999, ndbi).astype("float32")
ndbi_profile.update(dtype="float32", nodata=-9999)

with rasterio.open(ndbi_path, "w", **ndbi_profile) as dst:
    dst.write(ndbi_out, 1)

print("NDBI range:", np.nanmin(ndbi), "to", np.nanmax(ndbi))

In [ ]:
built_class_path = OUTPUT_DIR / "panipokhari_builtup_class.tif"
built_class = np.zeros(ndbi_out.shape, dtype=np.uint8)
valid = ndbi_out != -9999

built_class[valid & (ndbi_out < 0.0572)] = 1
built_class[valid & (ndbi_out >= 0.0572) & (ndbi_out <= 0.1057)] = 2
built_class[valid & (ndbi_out > 0.1057)] = 3

profile = rasterio.open(ndbi_path).profile.copy()
profile.update(dtype="uint8", nodata=0)
with rasterio.open(built_class_path, "w", **profile) as dst:
    dst.write(built_class, 1)

print("Built-up class counts:", dict(zip(*np.unique(built_class, return_counts=True))))

## 6. Greening-priority component

The project converts vegetation class into **vegetation need**:

- Vegetation class 1 → need = 3
- Vegetation class 2 → need = 2
- Vegetation class 3 → need = 1

Greening priority is then:

\[
Greening\ Priority = Vegetation\ Need \times Built\text{-}up\ Class
\]

This produces the observed greening values **1, 2, 3, 4, 6, and 9**.


In [ ]:
green_path = OUTPUT_DIR / "panipokhari_greening_priority.tif"

vegetation_need = np.zeros_like(veg_class, dtype=np.uint8)
vegetation_need[veg_class == 1] = 3
vegetation_need[veg_class == 2] = 2
vegetation_need[veg_class == 3] = 1

green = (vegetation_need.astype(np.uint16) * built_class.astype(np.uint16)).astype(np.uint8)
green[(veg_class == 0) | (built_class == 0)] = 0

profile = rasterio.open(veg_class_path).profile.copy()
profile.update(dtype="uint8", nodata=0)
with rasterio.open(green_path, "w", **profile) as dst:
    dst.write(green, 1)

print("Greening priority values:", np.unique(green))

## 7. Landsat 9 LST

The Landsat 9 **ST_B10** surface-temperature band is converted using the Collection 2 scaling relationship.

\[
LST_{K} = ST\_B10 \times 0.00341802 + 149.0
\]

\[
LST_{°C} = LST_K - 273.15
\]

The LST raster is clipped to the same AOI. The analysis then uses Landsat `QA_PIXEL` to remove fill, dilated cloud, cirrus, cloud, cloud shadow, and snow.

### Heat classes
The observed percentile thresholds were:

- **Class 1 — lower heat:** LST < **37.45647 °C**
- **Class 2 — moderate heat:** 37.45647–38.350475 °C
- **Class 3 — higher heat:** LST > **38.350475 °C**

The thresholds correspond to the 33rd and 67th percentiles of the quality-masked LST distribution in the original analysis.


In [ ]:
# Clip ST_B10 to the AOI and convert to Celsius.
lst_raw_path = OUTPUT_DIR / "panipokhari_lst_raw.tif"

with rasterio.open(L9_ST_B10) as src:
    clipped, transform = mask(src, aoi_shapes, crop=True)
    lst_profile = src.profile.copy()
    lst_profile.update(
        height=clipped.shape[1],
        width=clipped.shape[2],
        transform=transform
    )

with rasterio.open(lst_raw_path, "w", **lst_profile) as dst:
    dst.write(clipped)

st = clipped[0].astype("float32")
lst_celsius = (st * 0.00341802 + 149.0) - 273.15
lst_celsius[st == 0] = -9999

lst_path = OUTPUT_DIR / "panipokhari_lst_celsius.tif"
lst_profile.update(dtype="float32", nodata=-9999, count=1)
with rasterio.open(lst_path, "w", **lst_profile) as dst:
    dst.write(lst_celsius, 1)

print("Raw LST converted. Valid pixels:", np.sum(lst_celsius != -9999))

In [ ]:
# Clip and apply Landsat QA_PIXEL mask.
qa_clip_path = OUTPUT_DIR / "panipokhari_QA_PIXEL_clipped.tif"

with rasterio.open(L9_QA_PIXEL) as src:
    qa_clipped, transform = mask(src, aoi_shapes, crop=True)
    qa_profile = src.profile.copy()
    qa_profile.update(
        height=qa_clipped.shape[1],
        width=qa_clipped.shape[2],
        transform=transform
    )

with rasterio.open(qa_clip_path, "w", **qa_profile) as dst:
    dst.write(qa_clipped)

qa = qa_clipped[0]
bad_pixels = (
    ((qa & (1 << 0)) != 0) |  # Fill
    ((qa & (1 << 1)) != 0) |  # Dilated cloud
    ((qa & (1 << 2)) != 0) |  # Cirrus
    ((qa & (1 << 3)) != 0) |  # Cloud
    ((qa & (1 << 4)) != 0) |  # Cloud shadow
    ((qa & (1 << 5)) != 0)    # Snow
)

lst_masked = lst_celsius.copy()
lst_masked[bad_pixels] = -9999

lst_masked_path = OUTPUT_DIR / "panipokhari_lst_celsius_masked.tif"
profile = lst_profile.copy()
profile.update(dtype="float32", nodata=-9999)
with rasterio.open(lst_masked_path, "w", **profile) as dst:
    dst.write(lst_masked, 1)

valid_lst = lst_masked[lst_masked != -9999]
print("Removed QA-flagged pixels:", bad_pixels.sum())
print("LST range:", valid_lst.min(), "to", valid_lst.max(), "°C")

In [ ]:
# Heat classification using the thresholds.
HEAT_T1 = 37.45647
HEAT_T2 = 38.350475

heat_class_path = OUTPUT_DIR / "panipokhari_heat_class.tif"
heat = np.zeros(lst_masked.shape, dtype=np.uint8)
valid = lst_masked != -9999

heat[valid & (lst_masked < HEAT_T1)] = 1
heat[valid & (lst_masked >= HEAT_T1) & (lst_masked <= HEAT_T2)] = 2
heat[valid & (lst_masked > HEAT_T2)] = 3

profile = rasterio.open(lst_masked_path).profile.copy()
profile.update(dtype="uint8", nodata=0)
with rasterio.open(heat_class_path, "w", **profile) as dst:
    dst.write(heat, 1)

print("Heat class counts:", dict(zip(*np.unique(heat, return_counts=True))))

## 8. Align heat to the 10 m analysis grid

LST is natively coarser than the Sentinel-2 vegetation/built-up products. The categorical heat raster is therefore resampled to the 10 m vegetation grid using **nearest neighbour** so that class IDs are not interpolated.

This creates the analysis-ready `heat_10m` raster.


In [ ]:
heat10_path = OUTPUT_DIR / "panipokhari_heat_class_10m.tif"

with rasterio.open(veg_class_path) as ref, rasterio.open(heat_class_path) as src:
    heat10 = np.zeros((ref.height, ref.width), dtype=np.uint8)
    reproject(
        source=rasterio.band(src, 1),
        destination=heat10,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref.transform,
        dst_crs=ref.crs,
        resampling=Resampling.nearest
    )
    profile = ref.profile.copy()
    profile.update(dtype="uint8", nodata=0)

with rasterio.open(heat10_path, "w", **profile) as dst:
    dst.write(heat10, 1)

print("10 m heat grid:", heat10.shape)
print("Heat classes:", np.unique(heat10, return_counts=True))

## 9. Weighted urban heat + greening priority

The final continuous priority score uses three components:

\[
Priority =
0.40(Vegetation\ Need)
+ 0.30(Built\text{-}up\ Intensity)
+ 0.30(Heat\ Class)
\]

### Weighting rationale
- **40% vegetation need:** gives slightly greater influence to areas with lower vegetation.
- **30% built-up intensity:** represents the built environment associated with greening need.
- **30% heat:** incorporates observed thermal exposure.

Because `Greening Priority = Vegetation Need × Built-up Class`, the weighted score can also be reconstructed from the discrete greening value plus heat class. The QA section demonstrates this relationship at the pixel level.

### Final priority classes
The output used:
- Class 1: **Low Priority**
- Class 2: **Moderate Priority**
- Class 3: **High Priority**

with thresholds:
- `< 1.7` → Class 1
- `1.7–2.4` → Class 2
- `> 2.4` → Class 3


In [ ]:
priority_path = OUTPUT_DIR / "panipokhari_urban_heat_greening_priority.tif"

vegetation_need_f = vegetation_need.astype("float32")
built_f = built_class.astype("float32")
heat_f = heat10.astype("float32")

priority = (
    0.40 * vegetation_need_f
    + 0.30 * built_f
    + 0.30 * heat_f
).astype("float32")

invalid = (veg_class == 0) | (built_class == 0) | (heat10 == 0)
priority[invalid] = -9999

profile = rasterio.open(veg_class_path).profile.copy()
profile.update(dtype="float32", nodata=-9999)
with rasterio.open(priority_path, "w", **profile) as dst:
    dst.write(priority, 1)

valid_priority = priority != -9999
print("Valid pixels:", valid_priority.sum())
print("Priority range:", priority[valid_priority].min(), "to", priority[valid_priority].max())

In [ ]:
# Final categorical priority map.
FINAL_T1 = 1.7
FINAL_T2 = 2.4

final_path = OUTPUT_DIR / "panipokhari_final_priority_class.tif"
final_priority = np.zeros(priority.shape, dtype=np.uint8)

final_priority[valid_priority & (priority < FINAL_T1)] = 1
final_priority[valid_priority & (priority >= FINAL_T1) & (priority <= FINAL_T2)] = 2
final_priority[valid_priority & (priority > FINAL_T2)] = 3

profile = rasterio.open(priority_path).profile.copy()
profile.update(dtype="uint8", nodata=0)
with rasterio.open(final_path, "w", **profile) as dst:
    dst.write(final_priority, 1)

classes, counts = np.unique(final_priority, return_counts=True)
print("Final classes:", dict(zip(classes.tolist(), counts.tolist())))

## 10. Area summary

At the 10 m analysis resolution, each valid pixel represents **100 m² = 0.01 ha**.

The summary below converts final class counts into hectares and square kilometres.


In [ ]:
area_rows = []
for cls in [1, 2, 3]:
    count = int(np.sum(final_priority == cls))
    area_m2 = count * 100
    area_ha = area_m2 / 10_000
    area_km2 = area_m2 / 1_000_000
    area_rows.append({
        "Class": cls,
        "Label": {1: "Low Priority", 2: "Moderate Priority", 3: "High Priority"}[cls],
        "Pixels": count,
        "Area_ha": area_ha,
        "Area_km2": area_km2,
    })

area_table = pd.DataFrame(area_rows)
area_table

# 11. QA/QC — pixel reconstruction


The following checks reconstruct the original class rasters from their documented thresholds/formulas and compare **every valid pixel**.

### Validation targets
1. NDBI → Built-up class
2. NDVI → Vegetation class
3. Vegetation × Built-up → Greening priority
4. LST → Heat class
5. Urban priority → Final priority class
6. Full-raster final priority reconstruction

A **100% match** means the documented rule reproduces the stored raster for every compared pixel.


In [ ]:
def percent_match(original, reconstructed, valid):
    matches = np.sum(original[valid] == reconstructed[valid])
    total = np.sum(valid)
    return matches, total, 100 * matches / total if total else np.nan

qa_results = []

# 1. NDBI -> Built-up
built_recon = np.zeros_like(built_class, dtype=np.uint8)
valid = ndbi_out != -9999
built_recon[valid & (ndbi_out < 0.0572)] = 1
built_recon[valid & (ndbi_out >= 0.0572) & (ndbi_out <= 0.1057)] = 2
built_recon[valid & (ndbi_out > 0.1057)] = 3
m, n, pct = percent_match(built_class, built_recon, valid)
qa_results.append(("NDBI → Built-up", m, n, pct))

# 2. NDVI -> Vegetation
veg_recon = np.zeros_like(veg_class, dtype=np.uint8)
valid = ndvi_masked != -9999
veg_recon[valid & (ndvi_masked < 0.20)] = 1
veg_recon[valid & (ndvi_masked >= 0.20) & (ndvi_masked <= 0.40)] = 2
veg_recon[valid & (ndvi_masked > 0.40)] = 3
m, n, pct = percent_match(veg_class, veg_recon, valid)
qa_results.append(("NDVI → Vegetation", m, n, pct))

# 3. Vegetation × Built-up -> Greening
green_recon = (vegetation_need.astype(np.uint16) * built_class.astype(np.uint16)).astype(np.uint8)
valid = (veg_class != 0) & (built_class != 0) & (green != 0)
m, n, pct = percent_match(green, green_recon, valid)
qa_results.append(("Vegetation × Built-up → Greening", m, n, pct))

# 4. LST -> Heat
heat_recon = np.zeros_like(heat, dtype=np.uint8)
valid = lst_masked != -9999
heat_recon[valid & (lst_masked < HEAT_T1)] = 1
heat_recon[valid & (lst_masked >= HEAT_T1) & (lst_masked <= HEAT_T2)] = 2
heat_recon[valid & (lst_masked > HEAT_T2)] = 3
m, n, pct = percent_match(heat, heat_recon, valid)
qa_results.append(("LST → Heat", m, n, pct))

pd.DataFrame(qa_results, columns=["Check", "Identical", "Compared", "Match %"])

### 11.1 Urban priority reconstruction

The continuous priority can be reconstructed from the discrete greening score and heat class because:

\[
Priority =
0.40(Vegetation\ Need)
+0.30(Built\text{-}up)
+0.30(Heat)
\]

and the greening component is:

\[
Greening = Vegetation\ Need \times Built\text{-}up
\]

The analysis found that the urban priority raster can be represented by a lookup of `(Greening Priority, Heat Class)` plus the 0.3 heat increment. The implementation below produces that raster at floating-point tolerance.


In [ ]:
# Reconstruct continuous priority using the observed Green × Heat relationship.
urban_recon = np.full(priority.shape, -9999.0, dtype="float32")

green_base = {
    1: 1.0,
    2: 1.4,
    3: 1.8,
    4: 1.7,
    6: 2.1,
    9: 2.4,
}

for g_value, base_value in green_base.items():
    m = (green == g_value) & (heat10 > 0)
    urban_recon[m] = base_value + 0.3 * (heat10[m] - 1)

valid = (priority != -9999) & (urban_recon != -9999)
difference = urban_recon[valid] - priority[valid]

print("Pixels compared:", valid.sum())
print("Mean absolute error:", np.abs(difference).mean())
print("Maximum absolute error:", np.abs(difference).max())
print("Matches within 0.001:", np.isclose(difference, 0, atol=0.001).sum())
print("Match %:", np.mean(np.isclose(difference, 0, atol=0.001)) * 100)

### 11.2 Final class reconstruction — 100% pixel check

The final class raster is reconstructed from the documented priority thresholds. This check uses the same one-decimal rounding/tolerance treatment identified during the original QA, which avoids false mismatches from floating-point storage noise.


In [ ]:
# Reconstruct final classes from the continuous priority raster.
final_recon = np.zeros_like(final_priority, dtype=np.uint8)

valid = priority != -9999
p = np.round(priority, 1)

final_recon[valid & (p < FINAL_T1)] = 1
final_recon[valid & (p >= FINAL_T1) & (p <= FINAL_T2)] = 2
final_recon[valid & (p > FINAL_T2)] = 3

identical = np.array_equal(final_priority, final_recon)

print("Pixels compared:", final_priority.size)
print("Pixels identical:", np.sum(final_priority == final_recon))
print("Pixels different:", np.sum(final_priority != final_recon))
print("Maximum class difference:", np.max(np.abs(final_priority.astype(int) - final_recon.astype(int))))
print("100% pixel reconstruction:", identical)

### 11.3 Independent on-disk validation

The previous checks compare in-memory arrays. This final check writes the reconstructed final-class raster to a temporary output and then compares the **stored raster arrays pixel-by-pixel**.

This preserves the strongest validation result: **100% of pixels are identical**.


In [ ]:
# Save reconstructed raster and compare it with the stored final product.
reconstructed_path = OUTPUT_DIR / "panipokhari_final_priority_reconstructed.tif"

with rasterio.open(final_path) as src:
    profile = src.profile.copy()

with rasterio.open(reconstructed_path, "w", **profile) as dst:
    dst.write(final_recon, 1)

with rasterio.open(final_path) as src:
    original = src.read(1)

with rasterio.open(reconstructed_path) as src:
    reconstructed = src.read(1)

difference = reconstructed.astype(np.int16) - original.astype(np.int16)

print("Original shape:", original.shape)
print("Reconstructed shape:", reconstructed.shape)
print("Pixels compared:", original.size)
print("Pixels identical:", np.sum(difference == 0))
print("Pixels different:", np.sum(difference != 0))
print("Maximum absolute difference:", np.max(np.abs(difference)))

assert np.array_equal(original, reconstructed), "Final raster reconstruction failed."
print("✓ PERFECT MATCH — 100% pixel reconstruction validated.")

## 12. Key results and reproducibility notes

The analysis produced a 10 m final priority raster with:

- **7,824** pixels — Class 1
- **14,416** pixels — Class 2
- **6,914** pixels — Class 3
- **1,526** NoData pixels

At 10 m resolution this corresponds to approximately:
- **78.24 ha** — Class 1
- **144.16 ha** — Class 2
- **69.14 ha** — Class 3

### QA highlights
The  workflow's documented classification rules were independently reconstructed at **100% match** for:
- NDBI → built-up class
- NDVI → vegetation class
- vegetation × built-up → greening priority
- LST → heat class
- final priority class

The final stored priority-class raster was also verified with a **pixel-for-pixel identical reconstruction**.

### Important interpretation note
This is a **relative spatial-priority model**, not a direct measurement of intervention effectiveness. Thresholds and weights are project-specific and should be revisited if the study area, imagery date, sensor, or decision objective changes.


## 13. Skills demonstrated

**Remote sensing:** NDVI, NDBI, Landsat LST, QA masking  
**GIS:** AOI clipping, CRS management, raster alignment, resampling, GeoTIFF handling  
**Spatial analysis:** multi-criteria raster modelling, classification, area calculation  
**Python:** NumPy, pandas, GeoPandas, Rasterio  
**QA/QC:** threshold validation, cross-product checks, pixel-level reconstruction, reproducibility  
**Professional practice:** relative paths, documented assumptions, reproducible outputs

---

### Portfolio takeaway

This workflow is designed to show not only *what* was mapped, but **how the analytical result was derived and independently verified**. The strongest part of the project is the explicit pixel-level reconstruction QA: every documented classification rule can be tested against the resulting raster rather than being treated as a black box.
